# 02 — Análisis Exploratorio de Datos (EDA)

Análisis del dataset ENSO (1950–2026) publicado en **datos.gov.co**, enfocado en entender el comportamiento del Fenómeno El Niño / La Niña y su impacto en Colombia.


## 1. Preparación del entorno

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'enso_model_ready.csv'
REPORTS_PATH   = ROOT / 'reports'
REPORTS_PATH.mkdir(exist_ok=True)

COLORES_FASE = {'El Nino': '#E53935', 'La Nina': '#1E88E5', 'Neutral': '#8E8E8E'}


> Se importan pandas, numpy, matplotlib y seaborn. Se define una paleta de colores para las fases ENSO que se usará en todas las gráficas.


## 2. Carga del dataset

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Fecha'])

print(f"Filas    : {df.shape[0]:,}")
print(f"Columnas : {df.shape[1]}")
print(f"Periodo  : {df['Fecha'].min().strftime('%b %Y')}  →  {df['Fecha'].max().strftime('%b %Y')}")
print()
display(df.head())


> Se carga el dataset procesado. Contiene 914 registros mensuales con 17 variables, cubriendo enero 1950 a febrero 2026.


## 3. Estadísticas descriptivas

In [ ]:
cols = ['Temperatura_Pacifico_C', 'Temperatura_Base_C', 'Anomalia_C',
        'ONI_C', 'Anomalia_12m', 'Duracion_Meses']

display(df[cols].describe().round(2))


> Resumen de medidas de tendencia central y dispersión para las variables numéricas principales.
> La anomalía ONI oscila entre -2.03 y +2.75 °C, con una media cercana a 0, lo que indica que El Niño y La Niña se equilibran históricamente.


## 4. Distribución de variables numéricas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

variables = [
    ('Anomalia_C',             'Anomalía mensual ENSO (°C)'),
    ('ONI_C',                  'Índice ONI trimestral (°C)'),
    ('Temperatura_Pacifico_C', 'Temperatura Pacífico (°C)'),
    ('Temperatura_Base_C',     'Temperatura base climatológica (°C)'),
    ('Duracion_Meses',         'Duración del evento (meses)'),
    ('Anomalia_Delta',         'Variación mensual (°C)'),
]

for ax, (col, titulo) in zip(axes, variables):
    data = df[col].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white')
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media: {data.mean():.2f}')
    ax.axvline(data.median(), color='green', linestyle='--', linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.suptitle('Distribución de variables numéricas — ENSO 1950–2026', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_01_distribuciones.png', dpi=110, bbox_inches='tight')
plt.show()


> - `Anomalia_C` y `ONI_C` muestran distribución aproximadamente simétrica centrada en 0, con valores entre -2 y +3 °C.
> - `Temperatura_Pacifico_C` sigue una distribución normal con media alrededor de 27 °C.
> - `Duracion_Meses` tiene asimetría positiva: la mayoría de episodios duran menos de 20 meses, pero hay algunos que superan los 40.
> - `Temperatura_Base_C` es casi uniforme porque representa la media climatológica mensual.


## 5. Distribución por fase ENSO

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

variables_box = [
    ('Anomalia_C',    'Anomalía mensual (°C)'),
    ('ONI_C',         'Índice ONI (°C)'),
    ('Duracion_Meses','Duración del evento (meses)'),
]

orden = ['La Nina', 'Neutral', 'El Nino']
colores = [COLORES_FASE[f] for f in orden]

for ax, (col, titulo) in zip(axes, variables_box):
    grupos = [df.loc[df['Fase_Evento'] == f, col].dropna() for f in orden]
    bp = ax.boxplot(grupos, labels=orden, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_ylabel(titulo)

plt.suptitle('Distribución de variables por fase ENSO', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_02_boxplots_fase.png', dpi=110, bbox_inches='tight')
plt.show()


> Los boxplots confirman la separación clara entre fases: El Niño tiene valores positivos de ONI, La Niña negativos, y Neutral cerca de 0.
> La duración de los eventos activos (El Niño y La Niña) tiende a ser mayor que los periodos Neutrales.


## 6. Frecuencia de variables categóricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Fases
counts_fase = df['Fase_Evento'].value_counts().reindex(['Neutral', 'El Nino', 'La Nina'])
bars = axes[0].bar(counts_fase.index, counts_fase.values,
                   color=[COLORES_FASE[f] for f in counts_fase.index], edgecolor='white')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                 f'{int(bar.get_height())}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Meses por fase ENSO', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de meses')

# Intensidades
orden_int = ['Neutral', 'Debil', 'Moderado', 'Fuerte', 'Muy Fuerte']
counts_int = df['Intensidad_Evento'].value_counts().reindex(orden_int)
bars2 = axes[1].bar(counts_int.index, counts_int.values, color='steelblue', edgecolor='white')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                 f'{int(bar.get_height())}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Meses por intensidad', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Número de meses')
axes[1].tick_params(axis='x', rotation=15)

# Por década
dec_counts = df['Decada'].value_counts().sort_index()
bars3 = axes[2].bar(dec_counts.index, dec_counts.values, color='#546E7A', edgecolor='white')
for bar in bars3:
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('Registros por década', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Número de meses')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Frecuencia de variables categóricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_03_categoricas.png', dpi=110, bbox_inches='tight')
plt.show()


> La fase Neutral es la más frecuente con 419 meses (45.8%). El Niño y La Niña se distribuyen casi de igual forma con 249 y 246 meses respectivamente.
> La mayoría de los meses activos tienen intensidad Débil o Moderada. Los eventos Fuertes (57) y Muy Fuertes (18) son raros pero los de mayor impacto en Colombia.


## 7. Evolución temporal del índice ONI (1950–2026)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

# Sombrear áreas por fase
for _, row in df.iterrows():
    if row['Fase_Evento'] == 'El Nino':
        ax.axvspan(row['Fecha'], row['Fecha'] + pd.DateOffset(months=1),
                   alpha=0.3, color='#FFCDD2', linewidth=0)
    elif row['Fase_Evento'] == 'La Nina':
        ax.axvspan(row['Fecha'], row['Fecha'] + pd.DateOffset(months=1),
                   alpha=0.3, color='#BBDEFB', linewidth=0)

# Líneas de umbral ENSO
ax.axhline(0.5,  color='red',  linewidth=0.8, linestyle=':', alpha=0.6)
ax.axhline(-0.5, color='blue', linewidth=0.8, linestyle=':', alpha=0.6)
ax.axhline(0,    color='black', linewidth=0.7, linestyle='--', alpha=0.4)

# ONI y tendencia
ax.plot(df['Fecha'], df['ONI_C'], color='gray', linewidth=0.7, alpha=0.7, label='ONI mensual')
ax.plot(df['Fecha'], df['Anomalia_12m'], color='black', linewidth=2, label='Tendencia 12 meses')

# Años críticos Colombia
nino_co = {1982: 'El Niño 82', 1997: 'El Niño 97', 2015: 'El Niño 15'}
nina_co = {1999: 'La Niña 99', 2010: 'La Niña 10'}

for anio, etiq in nino_co.items():
    ax.axvline(pd.Timestamp(f'{anio}-01-01'), color='darkred', linewidth=1.5, linestyle='--')
    ax.text(pd.Timestamp(f'{anio}-01-01'), 2.8, etiq, rotation=90,
            color='darkred', fontsize=8, va='top', ha='right')

for anio, etiq in nina_co.items():
    ax.axvline(pd.Timestamp(f'{anio}-01-01'), color='darkblue', linewidth=1.5, linestyle='--')
    ax.text(pd.Timestamp(f'{anio}-01-01'), -1.9, etiq, rotation=90,
            color='darkblue', fontsize=8, va='bottom', ha='right')

ax.set_title('Índice ONI 1950–2026 — Episodios de impacto en Colombia', fontsize=12, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Anomalía (°C)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_04_serie_temporal.png', dpi=110, bbox_inches='tight')
plt.show()


> La serie muestra ciclos irregulares de 2 a 7 años, patrón característico del ENSO.
> Los episodios más intensos para Colombia se marcan con líneas verticales: El Niño 1982, 1997 y 2015 (sequías), La Niña 1999 y 2010 (inundaciones).
> La tendencia de 12 meses ayuda a ver el ciclo suavizando el ruido mensual.


## 8. Intensidad promedio por década

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ONI promedio por década y fase
dec_stats = (
    df[df['Fase_Evento'] != 'Neutral']
    .groupby(['Decada', 'Fase_Evento'])['ONI_C']
    .mean()
    .abs()
    .reset_index()
)

for fase, color in [('El Nino', '#E53935'), ('La Nina', '#1E88E5')]:
    sub = dec_stats[dec_stats['Fase_Evento'] == fase]
    axes[0].plot(sub['Decada'], sub['ONI_C'], marker='o', color=color,
                 linewidth=2, markersize=7, label=fase)

axes[0].set_title('ONI promedio absoluto por década', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Década')
axes[0].set_ylabel('|ONI| promedio (°C)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

# Eventos extremos por década
ext_dec = df.groupby('Decada')['Evento_Extremo'].sum().reset_index()
bars = axes[1].bar(ext_dec['Decada'], ext_dec['Evento_Extremo'],
                   color='#E53935', edgecolor='white', alpha=0.8)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Meses con evento extremo (|ONI| ≥ 1.5°C) por década', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Década')
axes[1].set_ylabel('Número de meses')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Evolución de la intensidad ENSO por década', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_05_intensidad_decada.png', dpi=110, bbox_inches='tight')
plt.show()


> El ONI promedio de El Niño muestra una tendencia creciente desde los 1960s, con los 2020s como la década más intensa del registro.
> Los eventos extremos se concentran en los 1990s y 2010s, décadas con los episodios de mayor impacto en Colombia.


## 9. Estacionalidad mensual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

MESES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
         'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

# ONI promedio por mes y fase
mes_fase = df.groupby(['Mes', 'Fase_Evento'])['ONI_C'].mean().unstack()

for fase, color in COLORES_FASE.items():
    if fase in mes_fase.columns:
        axes[0].plot(range(1, 13), mes_fase[fase], marker='o', color=color,
                     linewidth=2, markersize=6, label=fase)

axes[0].set_title('ONI promedio por mes', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('ONI promedio (°C)')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(MESES)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].legend()

# Frecuencia de fases por mes
mes_counts = (
    df.groupby(['Mes', 'Fase_Evento'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['El Nino', 'Neutral', 'La Nina'])
)

x = np.arange(12)
w = 0.28
for i, (fase, color) in enumerate([('El Nino', '#E53935'), ('Neutral', '#8E8E8E'), ('La Nina', '#1E88E5')]):
    axes[1].bar(x + (i - 1) * w, mes_counts[fase], width=w,
                color=color, alpha=0.8, edgecolor='white', label=fase)

axes[1].set_title('Frecuencia de fases por mes', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Número de meses históricos')
axes[1].set_xticks(x)
axes[1].set_xticklabels(MESES)
axes[1].legend()

plt.suptitle('Estacionalidad mensual del ENSO (1950–2026)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_06_estacionalidad.png', dpi=110, bbox_inches='tight')
plt.show()


> Los eventos El Niño suelen alcanzar su pico entre noviembre y enero, mientras que su inicio ocurre entre mayo y agosto.
> Esto es importante para Colombia: el pico de El Niño coincide con la segunda temporada de lluvias (sep–nov), reduciendo las precipitaciones. La Niña las intensifica en ese mismo periodo.


## 10. Matriz de correlaciones

In [ ]:
cols_corr = ['Temperatura_Pacifico_C', 'Temperatura_Base_C', 'Anomalia_C',
             'ONI_C', 'Anomalia_12m', 'Anomalia_Delta',
             'Duracion_Meses', 'Evento_Extremo', 'Anio']

corr = df[cols_corr].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, vmin=-1, vmax=1, ax=ax)
ax.set_title('Matriz de correlación — variables ENSO', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_07_correlaciones.png', dpi=110, bbox_inches='tight')
plt.show()


> Las correlaciones más altas del dataset:
> - `Anomalia_C` y `ONI_C`: r = 0.98 — casi perfecta, ya que el ONI es el promedio móvil de la anomalía mensual.
> - `ONI_C` y `Anomalia_12m`: r = 0.96 — la tendencia anual sigue de cerca el índice ONI.
> - `Anomalia_C` y `Evento_Extremo`: r = 0.83 — los meses más anómalos son los que definen eventos extremos.
> - `Duracion_Meses` con anomalías: r < 0.2 — la intensidad de un evento no predice cuánto va a durar.


## 11. Relaciones entre variables clave

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Anomalia_C vs ONI_C
for fase, color in COLORES_FASE.items():
    sub = df[df['Fase_Evento'] == fase]
    axes[0].scatter(sub['Anomalia_C'], sub['ONI_C'], color=color, alpha=0.3,
                    s=10, label=fase)
axes[0].set_title('Anomalía mensual vs ONI', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Anomalía mensual (°C)')
axes[0].set_ylabel('ONI (°C)')
axes[0].legend(fontsize=8)

# Temperatura_Pacifico_C vs ONI_C
for fase, color in COLORES_FASE.items():
    sub = df[df['Fase_Evento'] == fase]
    axes[1].scatter(sub['Temperatura_Pacifico_C'], sub['ONI_C'], color=color,
                    alpha=0.3, s=10, label=fase)
axes[1].set_title('Temperatura Pacífico vs ONI', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Temperatura Pacífico (°C)')
axes[1].set_ylabel('ONI (°C)')
axes[1].legend(fontsize=8)

# Duracion_Meses vs ONI_C
for fase, color in [('El Nino', '#E53935'), ('La Nina', '#1E88E5')]:
    sub = df[df['Fase_Evento'] == fase]
    axes[2].scatter(sub['Duracion_Meses'], sub['ONI_C'].abs(), color=color,
                    alpha=0.3, s=10, label=fase)
axes[2].set_title('Duración vs Intensidad del evento', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Duración (meses)')
axes[2].set_ylabel('|ONI| (°C)')
axes[2].legend(fontsize=8)

plt.suptitle('Relaciones entre variables clave por fase ENSO', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_08_scatter.png', dpi=110, bbox_inches='tight')
plt.show()


> El primer scatter confirma la relación casi perfecta entre anomalía mensual y ONI, con separación clara por fase.
> El segundo muestra que a mayor temperatura del Pacífico, mayor tendencia a El Niño.
> El tercero confirma que no hay una relación fuerte entre la duración y la intensidad del evento — un evento puede ser muy intenso y breve, o moderado y prolongado.


## 12. Episodios críticos para Colombia

In [ ]:
episodios = [
    ('1982-01', '1984-06', 'El Niño 1982–83 (Sequía severa)'),
    ('1997-01', '1999-06', 'El Niño 1997–98 (Más intenso s. XX)'),
    ('2009-06', '2012-06', 'La Niña 2010–11 (3.2M damnificados)'),
    ('2014-06', '2017-01', 'El Niño 2015–16 (Máximo histórico)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, (inicio, fin, titulo) in zip(axes, episodios):
    sub = df[(df['Fecha'] >= inicio) & (df['Fecha'] <= fin)]

    ax.plot(sub['Fecha'], sub['ONI_C'], color='black', linewidth=2)
    ax.fill_between(sub['Fecha'], sub['ONI_C'], 0,
                    where=(sub['ONI_C'] > 0), color='#E53935', alpha=0.35)
    ax.fill_between(sub['Fecha'], sub['ONI_C'], 0,
                    where=(sub['ONI_C'] < 0), color='#1E88E5', alpha=0.35)
    ax.axhline(0.5,  color='red',  linewidth=1, linestyle=':')
    ax.axhline(-0.5, color='blue', linewidth=1, linestyle=':')
    ax.axhline(0,    color='black', linewidth=0.7, linestyle='--')

    idx_max = sub['ONI_C'].abs().idxmax()
    ax.annotate(f"Pico: {sub.loc[idx_max, 'ONI_C']:.2f}°C",
                xy=(sub.loc[idx_max, 'Fecha'], sub.loc[idx_max, 'ONI_C']),
                xytext=(10, 12), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=9, fontweight='bold')

    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_ylabel('ONI (°C)')

plt.suptitle('Episodios ENSO de mayor impacto en Colombia', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_09_episodios_colombia.png', dpi=110, bbox_inches='tight')
plt.show()


> Los cuatro episodios de mayor impacto documentado en Colombia:
> - **El Niño 1982–83**: pico ~2.28 °C — sequías históricas y crisis en generación hidroeléctrica.
> - **El Niño 1997–98**: pico ~2.48 °C — el más intenso del siglo XX, sequías y emergencias nacionales.
> - **La Niña 2010–11**: pico ~−1.7 °C — "ola invernal", 3.2 millones de damnificados.
> - **El Niño 2015–16**: pico récord de +2.77 °C — el evento más intenso del periodo histórico.


## 13. Temperatura media del Pacífico por año

In [ ]:
temp_anual = df.groupby('Anio')['Temperatura_Pacifico_C'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(temp_anual['Anio'], temp_anual['Temperatura_Pacifico_C'],
        color='steelblue', linewidth=1.5, marker='o', markersize=3, label='Temp. media anual')

# Media del periodo completo
media_global = temp_anual['Temperatura_Pacifico_C'].mean()
ax.axhline(media_global, color='red', linewidth=1.2, linestyle='--',
           label=f'Media global: {media_global:.2f}°C')

ax.set_title('Temperatura media anual del Pacífico ecuatorial (1950–2026)', fontsize=12, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Temperatura (°C)')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_10_temperatura_anual.png', dpi=110, bbox_inches='tight')
plt.show()

print("Temperatura media por década:")
display(df.groupby('Decada')['Temperatura_Pacifico_C'].mean().round(3).reset_index())


> La temperatura media del Pacífico muestra variabilidad año a año, con los picos más altos coincidiendo con los grandes eventos El Niño (1983, 1998, 2016).
> La tabla por décadas muestra que los 2000s y 2010s tienen las temperaturas medias más altas del periodo histórico.


## 14. Hallazgos del EDA

### Variables más relacionadas entre sí

| Par de variables | Correlación | Interpretación |
|---|---|---|
| `Anomalia_C` ↔ `ONI_C` | 0.98 | El ONI es el promedio móvil de la anomalía mensual |
| `ONI_C` ↔ `Anomalia_12m` | 0.96 | La tendencia anual sigue de cerca el índice ONI |
| `Anomalia_C` ↔ `Evento_Extremo` | 0.83 | Los meses más anómalos definen los eventos extremos |
| `Temperatura_Pacifico_C` ↔ `Temperatura_Base_C` | 0.47 | Correlación moderada: la base explica parte de la TSM |
| `Duracion_Meses` ↔ `ONI_C` | < 0.2 | La intensidad no predice la duración del evento |

### Hallazgos clave para el storytelling Colombia

1. **El Niño 2015–16** es el episodio más intenso del registro (+2.77 °C). Mayor impacto en Colombia en el siglo XXI.
2. **La Niña 2010–11** fue el más devastador por inundaciones: 3.2 millones de damnificados pese a tener una anomalía menor que los grandes El Niño.
3. Los meses de **septiembre a noviembre** son los más críticos para Colombia: es cuando el ENSO alcanza su pico y coincide con la segunda temporada de lluvias.
4. Solo **75 meses** del periodo (8.2%) tienen eventos Fuertes o Muy Fuertes, pero en ellos se concentran los principales desastres del país.
5. Las décadas **2000s y 2010s** tienen las temperaturas medias del Pacífico más altas del registro histórico.

### Variables más relevantes para el modelo predictivo
`ONI_C`, `Anomalia_12m`, `Anomalia_Delta`, `Mes`, `Duracion_Meses` y `Temperatura_Pacifico_C` son las variables con mayor capacidad para predecir la fase e intensidad del ENSO.
